In [21]:
import pandas as pd
import json

from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR,NuSVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd
from helpers.modeling import (
    identify_column_types,
    create_preprocessor,
    evaluate_model,
)


with open('results.json', 'r') as f:
    results = json.load(f)

In [8]:
df = pd.read_csv("../Datasets/processed/UHPC_dataset/semantic_recoding_features_50_with_publications.csv")
df.head()

,cement,cement_type,silica_fume,fly_ash,fly_ash_type,limestone_powder,quartz_powder,glass_powder,rice_husk_ash,metakaolin,...,fiber1_diameter,fiber2_type,fiber2_amount,water,sp_type,sp_amount,curing_method,curing_temp,cs_28d,paper_reference
0,839.0,OPC_I,104.0,104.0,class F,0.0,0.0,0.0,0.0,0.0,...,0.2,not_applicable,0.0,147.0,PCE_SP,56.50,Standard Curing,NaN,135.0,Ref-1-data
1,839.0,OPC_I,104.0,52.0,class F,0.0,0.0,0.0,0.0,0.0,...,0.2,not_applicable,0.0,147.0,PCE_SP,59.33,Standard Curing,NaN,132.0,Ref-1-data
2,839.0,OPC_I,104.0,26.0,class F,0.0,0.0,0.0,0.0,0.0,...,0.2,not_applicable,0.0,147.0,PCE_SP,59.33,Standard Curing,NaN,122.5,Ref-1-data
3,839.0,OPC_I,104.0,0.0,not_applicable,0.0,0.0,0.0,0.0,0.0,...,0.2,not_applicable,0.0,147.0,PCE_SP,62.15,Standard Curing,NaN,116.0,Ref-1-data
4,839.0,OPC_I,104.0,52.0,class F,0.0,0.0,0.0,0.0,52.0,...,0.2,not_applicable,0.0,147.0,PCE_SP,64.98,Standard Curing,NaN,134.0,Ref-1-data


In [9]:
X = df.drop(columns="cs_28d")
y = df["cs_28d"]

pub_col = "paper_reference"  
held_out_pub = "Ref-1-data"

train_mask = df[pub_col] != held_out_pub
test_mask  = df[pub_col] == held_out_pub

X_train, y_train = X[train_mask], y[train_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

In [11]:
numerical_cols, one_hot_columns, k_fold_columns = identify_column_types(X)

preprocessor = create_preprocessor(numerical_cols, one_hot_columns, k_fold_columns, 
                                   handle_unknown='ignore')

In [24]:
def run_pipeline(model_cls, model_key, kernel_kwargs=None):
    params = results["best_params"]["best_params_publications_included"][model_key]
    pipeline = Pipeline([('preprocessor', preprocessor),
                          ('model', model_cls(**(kernel_kwargs or {})))])
    pipeline.set_params(**params)
    pipeline.fit(X_train, y_train)

    train_metrics = evaluate_model(y_train, pipeline.predict(X_train), set_name='train')
    test_metrics = evaluate_model(y_test, pipeline.predict(X_test), set_name='test')
    return pipeline, train_metrics, test_metrics

In [25]:
knn_pipeline, knn_train, knn_test = run_pipeline(KNeighborsRegressor, "knn")



TRAIN SET PERFORMANCE
RMSE: 7.4427
MAE: 5.0768
R2: 0.9583
Correlation: 0.9795

TEST SET PERFORMANCE
RMSE: 8.8787
MAE: 6.1647
R2: -0.4148
Correlation: 0.3949


In [26]:
svr_pipeline, svr_train, svr_test = run_pipeline(SVR, "svr", {"kernel": "rbf"})



TRAIN SET PERFORMANCE
RMSE: 7.5987
MAE: 4.9105
R2: 0.9565
Correlation: 0.9781

TEST SET PERFORMANCE
RMSE: 26.4061
MAE: 22.6734
R2: -11.5143
Correlation: -0.5773


In [27]:
nusvr_pipeline, nusvr_train, nusvr_test = run_pipeline(NuSVR, "nusvr", {"kernel": "rbf"})


TRAIN SET PERFORMANCE
RMSE: 8.2266
MAE: 5.6340
R2: 0.9490
Correlation: 0.9743

TEST SET PERFORMANCE
RMSE: 27.9044
MAE: 26.8461
R2: -12.9747
Correlation: 0.0060
